In [1]:
import math
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df      = pd.read_csv('data/cox2_th_final/output/cox2_th-meta-clean.tsv', sep='\t')
cmap_df = pd.read_csv('colormap-oom-order.csv')
colormap = dict(zip(cmap_df['order'], cmap_df['color']))

# --- Configurable ---
OUTER_RANK = 'order'
FONT_SIZE  = 12
FIG_W, FIG_H = 1000, 600   # must match the figure size below
# --------------------

df[OUTER_RANK] = df[OUTER_RANK].fillna('Unknown')

counts = (
    df.groupby([OUTER_RANK, 'genus'])
    .size()
    .reset_index(name='count')
)

outer_vals   = sorted(counts[OUTER_RANK].unique())
# Fall back to grey for any value not in the colormap
outer_colors = {v: colormap.get(v, '#aaaaaa') for v in outer_vals}

fig = px.treemap(
    counts,
    path=[OUTER_RANK, 'genus'],
    values='count',
    title=f'Database composition by {OUTER_RANK} and genus',
    color=OUTER_RANK,
    color_discrete_map=outer_colors,
)

labels    = list(fig.data[0].labels)
values    = list(fig.data[0].values)
parents   = list(fig.data[0].parents)
outer_set = set(outer_vals)
total     = counts['count'].sum()

CHAR_W  = FONT_SIZE * 0.6
LINE_H  = FONT_SIZE * 1.8

def tile_text(lbl, par, val):
    if lbl in outer_set or par == '':
        return ''
    count = int(val)
    lines = [lbl, str(count)]
    text_w = max(len(l) for l in lines) * CHAR_W
    text_h = len(lines) * LINE_H
    frac = val / total
    tile_area = frac * FIG_W * FIG_H
    tile_w = math.sqrt(tile_area * 2)
    tile_h = math.sqrt(tile_area / 2)
    if tile_w >= text_w and tile_h >= text_h:
        return f'{lbl}<br>{count}'
    return ''

tile_texts = [tile_text(lbl, par, val) for lbl, par, val in zip(labels, parents, values)]

fig.update_traces(
    text=tile_texts,
    textinfo='text',
    textfont=dict(size=FONT_SIZE),
    insidetextfont=dict(size=FONT_SIZE),
    outsidetextfont=dict(size=FONT_SIZE),
    hovertemplate='<b>%{label}</b><br>Sequences: %{value:.0f}<extra></extra>',
    marker=dict(pad=dict(t=0, l=3, r=3, b=3)),
)

for val in outer_vals:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color=outer_colors[val], symbol='square'),
        name=val,
        showlegend=True,
    ))

fig.update_layout(
    width=FIG_W,
    height=FIG_H,
    margin=dict(t=50, l=10, r=10, b=10),
    showlegend=True,
    legend=dict(title=OUTER_RANK.capitalize(), itemsizing='constant'),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
)

fig.show()


In [2]:
fig.write_image("treemap.svg")

/tmp/ipykernel_2102559/2107151457.py:1: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


